# STEP 1 — Overview (Markdown Description)
# This script updates ESP-r latitude configuration files using EPC dataset values.
#
# What it does:
# 1. Loads the EPC dataset CSV file.
# 2. Extracts BUILDING_REFERENCE_NUMBER and LATITUDE values.
# 3. Iterates through each building folder in the baseline_models directory.
# 4. Searches for a "cfg/latitude.txt" file inside each archetype folder.
# 5. If found, replaces all instances of "LAT" with the corresponding LATITUDE value.
# 6. Saves the updated file.
# 7. Skips any buildings or files that are missing.
#
# Notes:
# - Safe to re-run (only replaces "LAT").
# - Skips missing files silently with logging.
# - Designed for Jupyter Notebook step-by-step execution.

In [1]:
# STEP 2 — Import Libraries
# Import required Python libraries

import os
import pandas as pd

In [ ]:
# STEP 3 — Load EPC Dataset
# Load the CSV file and prepare lookup dictionary

csv_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR_update.csv"

df = pd.read_csv(csv_path)

# Ensure correct data types
df["BUILDING_REFERENCE_NUMBER"] = df["BUILDING_REFERENCE_NUMBER"].astype(str)

# Create lookup dictionary: {building_id: latitude}
lat_lookup = dict(zip(df["BUILDING_REFERENCE_NUMBER"], df["LATITUDE"]))

print(f"Loaded {len(lat_lookup)} building latitude entries.")

Loaded 117 building latitude entries.


In [3]:
# STEP 4 — Define Base Directory
# Set the path to the baseline_models directory

base_dir = "/Users/jakegehrung/Desktop/data_raw/baseline_models"

# List all building folders
building_folders = [f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f))]

print(f"Found {len(building_folders)} building folders.")

Found 117 building folders.


In [4]:
# STEP 5 — Process Each Building Folder
# Iterate through buildings and update latitude.txt files

updated_count = 0
skipped_count = 0

for building_id in building_folders:
    building_path = os.path.join(base_dir, building_id)
    
    # Check if building exists in EPC dataset
    if building_id not in lat_lookup:
        skipped_count += 1
        continue
    
    latitude_value = str(lat_lookup[building_id])
    
    # Loop through archetype folders
    for subfolder in os.listdir(building_path):
        subfolder_path = os.path.join(building_path, subfolder)
        
        if not os.path.isdir(subfolder_path):
            continue
        
        cfg_path = os.path.join(subfolder_path, "cfg")
        latitude_file = os.path.join(cfg_path, "latitude.txt")
        
        # Check if latitude.txt exists
        if os.path.exists(latitude_file):
            try:
                with open(latitude_file, "r") as file:
                    content = file.read()
                
                # Replace LAT with actual latitude
                updated_content = content.replace("LAT", latitude_value)
                
                with open(latitude_file, "w") as file:
                    file.write(updated_content)
                
                updated_count += 1
            
            except Exception as e:
                print(f"Error processing {latitude_file}: {e}")
                skipped_count += 1

print("Processing complete.")
print(f"Files updated: {updated_count}")
print(f"Files skipped: {skipped_count}")

Processing complete.
Files updated: 1
Files skipped: 0


In [5]:
# STEP 6 — Optional Verification
# Check a sample file to confirm replacement worked

sample_building = building_folders[0]

sample_path = os.path.join(base_dir, sample_building)

for subfolder in os.listdir(sample_path):
    latitude_file = os.path.join(sample_path, subfolder, "cfg", "latitude.txt")
    
    if os.path.exists(latitude_file):
        print(f"\nSample check: {latitude_file}")
        with open(latitude_file, "r") as file:
            print(file.read())
        break